In [ ]:
from math import sqrt
import pandas as pd
import numpy as np

In [ ]:
import scipy

##Test de Welch

#Estadístico T
def T(X,Y):
    n = len(X)
    m = len(Y)

    numerador = X.mean() - Y.mean()

    denominador = sqrt((X.var()/n)+(Y.var()/m))

    return numerador/denominador


#Nu (grados de libertad de la distribución)

def nu(X,Y):
    n = len(X)
    m = len(Y)

    S2X_n = X.var()/n
    S2Y_m = Y.var()/m


    numerador = (S2X_n + S2Y_m)**2

    denominador = S2X_n**2 /(n-1) + S2Y_m**2 / (m-1)

    return numerador / denominador

#Test de Welch

def Welch(X,Y, alpha):
    est = T(X,Y)
    Nu = nu(X,Y)

    p = 2*(1- scipy.stats.t.cdf(abs(est), df=Nu))

    R = abs(est) >= scipy.stats.t.ppf(1-alpha/2, df=Nu)

    return (est,p,R)

In [ ]:
##Test Wilcoxon

def Wilcoxon(X,Y, alpha):


    #Creo una tabla que diga su rango junto a grupo incial x o y:
    datos = np.concatenate([X, Y])
    grupos = ["x"] * len(X) + ["y"] * len(Y)

    df = pd.DataFrame({"Dato": datos, "Grupo": grupos})

    df = df.sort_values(by="Dato").reset_index(drop=True)

    df["rango"] = df.index + 1

    W = df[df["Grupo"] == "x"]["rango"].sum()

    return (W,"p-value","Bool") #NO SE CALCULAR LA REGION AYUDA

In [ ]:
#Test Mann–Whitney }


def MannWhitney(X,Y, alpha):

    n = len(X)
    m = len(Y)
    N = n + m

    #Creo una tabla que diga su rango junto a grupo incial x o y:
    datos = np.concatenate([X, Y])
    grupos = ["x"] * n + ["y"] * m

    df = pd.DataFrame({"Dato": datos, "Grupo": grupos})

    df = df.sort_values(by="Dato").reset_index(drop=True)

    df["rango"] = df.index + 1

    #Calculo de los estadísticos principales
    W_x = df[df["Grupo"] == "x"]["rango"].sum()
    W_y = df[df["Grupo"] == "y"]["rango"].sum()

    U_x = n*m + n*(n+1)/2 - W_x

    U_y = n*m + m*(m+1)/2 - W_y

    U = min(U_x,U_y)

    E = n*m/2

    #Agrupa los datos, anota la frecuencia y rescatamos el array de repeticiones
    t = df.groupby("Dato").size().to_numpy()

    suma = np.sum(t**3 -t)

    var_U = (N+1 - (suma/(N*(N-1))))*E/6

    epsilon = (U-E)/sqrt(var_U)

    Rechazo = abs(epsilon) >= scipy.stats.norm.ppf(alpha/2)

    p = 2*(1- scipy.stats.norm.cdf(abs(epsilon)))





    return (epsilon,p, Rechazo)